In [ ]:
!pip install mxnet
!pip install gluonnlp pandas tqdm
!pip install sentencepiece
!pip install transformers
!pip install torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.1/49.1 MB 14.4 MB/s eta 0:00:00
  Attempting uninstall: graphviz
    Found existing installation: graphviz 0.20.3
    Uninstalling graphviz-0.20.3:
      Successfully uninstalled graphviz-0.20.3
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 344.5/344.5 kB 7.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for gluonnlp: filename=gluonnlp-0.10.0-cp310-cp310-linux_x86_64.whl size=661658 sha256=29254a564c5538e8376478fe6620b855f3ac91605fc95d8ec336b1af5242c23a
  Stored in directory: /root/.cache/pip/wheels/1a/1e/0d/99f55911d90f2b95b9f7c176d5813ef3622894a4b30fde6bd3
Successfully built gluonnlp


In [ ]:
from sentence_transformers import SentenceTransformer, util
import numpy as np
from google.colab import drive
import pandas as pd

drive.mount('/content/drive')
file_path = '/content/drive/My Drive/egg/meaningsWithSentences.csv'
data = pd.read_csv(file_path)

# Fill missing values in the meaning columns with empty strings to handle NaN values
#data[['meaning1', 'meaning2', 'meaning3', 'meaning4']] = data[['meaning1', 'meaning2', 'meaning3', 'meaning4']].fillna('')

model = SentenceTransformer('snunlp/KR-SBERT-V40K-klueNLI-augSTS')

# Fill missing values in the meaning columns with empty strings to handle NaN values
data[['meaning1', 'meaning2', 'meaning3', 'meaning4']] = data[['meaning1', 'meaning2', 'meaning3', 'meaning4']].fillna('')

# Combine all meanings into a list for each row
data['all_meanings'] = data.apply(lambda row: [row['meaning1'], row['meaning2'], row['meaning3'], row['meaning4']], axis=1)

# Convert all sentences and meanings to embeddings in batch
sentences = data[['sentence1', 'sentence2', 'sentence3']].fillna('').values.flatten()
meanings = data['all_meanings'].explode().dropna().unique()

sentence_embeddings = model.encode(sentences, convert_to_tensor=True)
meaning_embeddings = model.encode(meanings, convert_to_tensor=True)

# Create a dictionary to store meaning embeddings for faster lookup
meaning_embedding_dict = {meaning: embedding for meaning, embedding in zip(meanings, meaning_embeddings)}

# Reshape sentence embeddings back to the original structure
sentence_embeddings = sentence_embeddings.reshape(len(data), 3, -1)

# Define function to calculate the maximum cosine similarity between a sentence and multiple meanings
def calculate_max_similarity(sentence_embedding, meanings, meaning_embedding_dict):
    meanings_embeddings = [meaning_embedding_dict[meaning] for meaning in meanings if meaning.strip() != '']
    if not meanings_embeddings:
        return np.nan
    # Calculate cosine similarity
    cosine_scores = [util.pytorch_cos_sim(sentence_embedding, meaning_embedding) for meaning_embedding in meanings_embeddings]
    # Return the maximum similarity score
    return float(max(cosine_scores))

# Iterate over each row to calculate similarities
similarities = []
for idx, row in data.iterrows():
    row_similarities = []
    for i in range(3):  # for sentence1, sentence2, sentence3
        sentence_embedding = sentence_embeddings[idx, i]
        if sentence_embedding is None or sentence_embedding.nelement() == 0:
            row_similarities.append(np.nan)
        else:
            max_similarity = calculate_max_similarity(sentence_embedding, row['all_meanings'], meaning_embedding_dict)
            row_similarities.append(max_similarity)
    similarities.append(row_similarities)

# Add the calculated similarities to the DataFrame
data[['sentence1Similarity', 'sentence2Similarity', 'sentence3Similarity']] = pd.DataFrame(similarities, index=data.index)

output_path = '/content/drive/My Drive/egg/SentenceBert_data-5.csv'
data.to_csv(output_path, index=False)

# 문장 임베딩 추출 방법
# - 단어에 대한 임베딩을 각각 구한 후, 임베딩의 평균 사용
# - KR-SBERT(Sentence BERT)를 통한 문장 임베딩 추출


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
   symbol_id tts_expression   meaning1   meaning2     meaning3 meaning4  \
0          1        TV보고싶어요  TV 보고 싶어요  티비 보고 싶어요  텔레비전 보고 싶어요            
1          3      가방(실내화가방)         가방      실내화가방                         
2          4             간식         간식                                    
3          5             간식         간식                                    
4          6          감사합니다      감사합니다                                    

   sentence1 sentence2 sentence3                           all_meanings  \
0  TV 보고 싶어요  TV 보고 싶다  TV 보고 싶어  [TV 보고 싶어요, 티비 보고 싶어요, 텔레비전 보고 싶어요, ]   
1         가방        쇼핑        물건                        [가방, 실내화가방, , ]   
2        케이크         빵       디저트                             [간식, , , ]   
3         쿠키       케이크       디저트                             [간식, , , ]   
4      반갑습니다  만나서 반가워요     안녕하세요             